![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and Mistral `mistral-small-3-1-24b-instruct-2503` to analyze eXtensive Business Reporting Language (XBRL) tags of financial reports

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of tag entity extraction in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to use `mistral-small-3-1-24b-instruct-2503` model to analyze XBRL tags of financial phrases.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
4. [Analyze the sentiment](#Analyze-the-sentiment)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install datasets | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see
[documentation](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui).

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, project_id=project_id)

<a id="Data-loading"></a>
## Data loading

Download the `nlpaueb/finer-139` dataset.

In [5]:
from datasets import load_dataset

finer_train = load_dataset(
    "nlpaueb/finer-139", split="train", revision="refs/convert/parquet"
)

Retrieve the entity tags.

In [6]:
ner_tags = finer_train.features["ner_tags"].feature.names
ner_tags[:10]

['O',
 'B-AccrualForEnvironmentalLossContingencies',
 'B-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 'I-AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife',
 'B-AllocatedShareBasedCompensationExpense',
 'B-AmortizationOfFinancingCosts',
 'B-AmortizationOfIntangibleAssets',
 'I-AmortizationOfIntangibleAssets',
 'B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 'I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount']

Gather data, which references a single tag

In [7]:
single_tag_indexes: dict[int, int] = {}

for i, tags in enumerate(finer_train["ner_tags"]):
    tag_count = sum(tag != 0 for tag in tags)
    if tag_count == 1:
        single_tag_indexes[i] = next(item for item in tags if item)

list(single_tag_indexes.items())[:20]

[(26, 168),
 (34, 14),
 (41, 41),
 (43, 41),
 (45, 41),
 (72, 87),
 (73, 87),
 (74, 87),
 (84, 41),
 (86, 41),
 (89, 41),
 (90, 41),
 (92, 41),
 (93, 41),
 (125, 93),
 (126, 93),
 (127, 72),
 (170, 67),
 (194, 67),
 (201, 85)]

Convert the tokens into sequences for the model

In [8]:
sequences = [
    " ".join(token)
    for i, token in enumerate(finer_train["tokens"])
    if i in single_tag_indexes
]

Inspect exemplary sequence.

In [9]:
sequences[3]

'Interest on the 7.00 % Senior Notes is due semi - annually .'

Select test sequences

In [10]:
from collections import defaultdict

sequences_for_tag: defaultdict[str, list[str]] = defaultdict(list)
for sequence, tag_index in zip(sequences, single_tag_indexes.values()):
    tag_value = ner_tags[tag_index]

    sequences_for_tag[tag_value].append(sequence)

selected_ner_tags = list(sequences_for_tag)[:10]

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

In [11]:
for model in api_client.foundation_models.ChatModels:
    print(f"- {model}")

- ibm/granite-4-h-small
- meta-llama/llama-3-3-70b-instruct
- meta-llama/llama-4-maverick-17b-128e-instruct-fp8
- mistral-large-2512
- mistralai/mistral-medium-2505
- mistralai/mistral-small-3-1-24b-instruct-2503
- openai/gpt-oss-120b


You need to specify `model_id` that will be used for inferencing:

In [12]:
model_id = api_client.foundation_models.ChatModels.MISTRAL_SMALL_3_1_24B_INSTRUCT_2503

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenChatParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [13]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {
    GenParams.TEMPERATURE: 0,
    GenParams.REPETITION_PENALTY: 1,
}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [14]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(model_id=model_id, params=parameters, api_client=api_client)

### Model's details

In [15]:
model.get_details()

{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503',
 'label': 'mistral-small-3-1-24b-instruct-2503',
 'provider': 'Mistral AI',
 'source': 'Hugging Face',
 'indemnity': 'NON_IBM',
 'functions': [{'id': 'autoai_rag'},
  {'id': 'autoai_sql_rag'},
  {'id': 'image_chat'},
  {'id': 'text_chat'},
  {'id': 'text_generation'}],
 'short_description': 'This model is an instruction-finetuned version of: Mistral-Small-3.1-24B-Base-2503.',
 'long_description': 'Mistral Small 3 (2501), Mistral Small 3.1 (2503) adds state-of-the-art vision understanding and enhances long context capabilities up to 128k tokens without compromising text performance. With 24 billion parameters, this model achieves top-tier capabilities in both text and vision tasks.',
 'terms_url': 'https://www.apache.org/licenses/LICENSE-2.0',
 'input_tier': 'class_c1',
 'output_tier': 'class_17',
 'number_params': '24b',
 'min_shot_size': 1,
 'task_ids': ['question_answering',
  'summarization',
  'retrieval_augmented_genera

<a id="Analyze-the-sentiment"></a>
## Analyze the sentiment

Define instructions for the model. 

**HINT:** All possible tags must be attached in the instruction

In [16]:
def get_messages(
    example_reports: list[str],
    example_tags: list[str],
    all_tags: list[str],
    report_to_check: str,
) -> list[dict[str, str]]:
    system_content = (
        "You are a tag classification model. "
        "Determine the eXtensive Business Reporting Language tag in the financial report from following tags. "
        "The prefix `B` means it is a balance sheet item, while prefix `I` means it is an income statement item. "
        "The following tags are available, produce only a single tag:\n"
    )
    system_content += "\n".join(f"- {tag}" for tag in all_tags)

    messages = [{"role": "system", "content": system_content}]

    for report, tag in zip(example_reports, example_tags):
        messages.append({"role": "user", "content": report})
        messages.append({"role": "assistant", "content": tag})

    messages.append({"role": "user", "content": report_to_check})

    return messages

Prepare few-shot examples.

In [17]:
example_tags: list[str] = sorted(selected_ner_tags)
example_reports: list[str] = [sequences_for_tag[tag][0] for tag in example_tags]

print(example_tags)
print(*example_reports, sep="\n")

['B-AreaOfRealEstateProperty', 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired', 'B-DebtInstrumentInterestRateStatedPercentage', 'B-FiniteLivedIntangibleAssetUsefulLife', 'B-GuaranteeObligationsMaximumExposure', 'B-LineOfCreditFacilityCurrentBorrowingCapacity', 'B-LineOfCreditFacilityMaximumBorrowingCapacity', 'B-LossContingencyDamagesSoughtValue', 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross', 'B-UnrecognizedTaxBenefits']
The East Rutherford facility consisted of warehouses and offices totaling approximately 81,000 square feet of space .
In 2010 , the Rialto segment acquired indirectly 40 % managing member equity interests in two limited liability companies ( " LLCs " ) in partnership with the FDIC ( “ FDIC Portfolios ” ) .
Rialto used the net proceeds of the 7.00 % Senior Notes to provide additional working capital for RMF , and to make investments in the funds that Rialto manages , as well as for general corporate purposes .
Patent

### Analyze financial phrase eXtensive Business Reporting Language using `mistral-small-3-1-24b-instruct-2503` model

Analyze the report

In [18]:
results: list[str] = []
for tag in selected_ner_tags:
    messages = get_messages(
        example_reports,
        example_tags,
        selected_ner_tags,
        sequences_for_tag[tag][1],
    )

    response = model.chat(messages)
    results.append(response["choices"][0]["message"]["content"])

results

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 "I'm sorry, but the provided text does not contain any of the specified tags. It discusses the maturity dates of a credit facility, which is not covered by the given tags.",
 'I do not have a tag for this.',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

<a id="Score-the-model"></a>
## Score the model

**Note:** To run the Score section for model scoring please transform following `markdown` cells to `code` cells.
Have in mind that the score is calculated only on dataset sample, for relevant performance metric please score the model on the whole `nlpaueb/finer-139` dataset.

Get the true labels.

In [19]:
y_true = selected_ner_tags
y_true

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 'B-LineOfCreditFacilityMaximumBorrowingCapacity',
 'B-LossContingencyDamagesSoughtValue',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

Get the sentiment labels returned by the `mistral-small-3-1-24b-instruct-2503` model.

In [20]:
y_pred = [result.strip() for result in results]
y_pred

['B-UnrecognizedTaxBenefits',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-DebtInstrumentInterestRateStatedPercentage',
 "I'm sorry, but the provided text does not contain any of the specified tags. It discusses the maturity dates of a credit facility, which is not covered by the given tags.",
 'I do not have a tag for this.',
 'B-GuaranteeObligationsMaximumExposure',
 'B-FiniteLivedIntangibleAssetUsefulLife',
 'B-LineOfCreditFacilityCurrentBorrowingCapacity',
 'B-ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsGrantsInPeriodGross',
 'B-AreaOfRealEstateProperty']

Calculate accuracy score.

In [21]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_pred, y_true))

0.8


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to predict the financial phrases XBRL tag with `mistral-small-3-1-24b-instruct-2503` on watsonx. 

Check out our _[Online Documentation](https://ibm.github.io/watsonx-ai-python-sdk/samples.html#)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.